# Reward Shaping SolarSystemLander

Colab L4 runner for ground thrust penalty reward shaping experiments.

## Set up

In [ ]:
# cell: colab-setup

!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt
%pip install -r reward_shaping/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")
sys.path.insert(0, "reward_shaping/src")

from hpo.notebook.colab import setup_colab

COLAB = setup_colab()

In [ ]:
# cell: reward-shaping-setup; requires: colab-setup

from datetime import datetime, timezone
from pathlib import Path
import shutil

import torch
from dqn.vector_training import VectorTrainer, VectorTrainingConfig
from hpo.solar_system_lander.environment import EnvFactory, World, worlds_by_name
from reward_shaping import make_reward_shaping_vector_env
from reward_shaping.experiment_harness import (
    historical_score,
    load_q_net_checkpoint,
    prepare_run_directory,
    robust_score,
    save_q_net_checkpoint,
    write_eval_scores,
    write_yaml,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

## Configure run

In [ ]:
# cell: run-config; requires: reward-shaping-setup

OBSERVATION_MODE = "10d"
STUDY_SERIES_NAME = f"solar_system_lander_{OBSERVATION_MODE}_elise_accel"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_ground_thrust_penalty")

GROUND_THRUST_PENALTY = 0.5
ADDITIONAL_TRAINING_EPISODES = 500
NUM_ENVS = 22
ROBUST_EPISODES_PER_WORLD = 50
REPLAY_MEMORY_CAPACITY = 76_754
SOURCE_EPS_END = 0.04400876385215351
SOURCE_EPS_DECAY = 38_793
SOURCE_LEARNING_RATE = 0.0006229370728793535

INITIAL_CHECKPOINT_DRIVE = COLAB.drive_study_dir / "best_checkpoints" / STUDY_SERIES_NAME / "best_eval_checkpoint.pt"
LOCAL_RUN_ROOT = Path("/content/rl_lab/reward_shaping/runs")
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/rl_lab/reward_shaping/runs")

# Keep the run close to the 253 checkpoint's HPO region.
# Since this fine-tunes a trained checkpoint, start exploration near the source eps_end instead of 1.0.
TRAINING_CONFIG = VectorTrainingConfig(
    num_episodes=ADDITIONAL_TRAINING_EPISODES,
    eps_start=2*SOURCE_EPS_END,
    eps_end=SOURCE_EPS_END,
    eps_decay=SOURCE_EPS_DECAY,
    learning_rate=SOURCE_LEARNING_RATE,
    batch_size=512,
    gamma=0.995,
    tau=0.002,
    learning_starts=2500,
    optimize_every=2,
    hidden_size=128,
)

TRAINING_FACTORY = EnvFactory(
    OBSERVATION_MODE,
    worlds=worlds_by_name(
        World.MERCURY,
        World.VENUS,
        World.VENUS,
        World.VENUS,
        World.VENUS,
        World.EARTH,
        World.EARTH,
        World.EARTH,
        World.EARTH,
        World.MOON,
        World.MARS,
    ),
)
EVALUATION_FACTORY = EnvFactory(OBSERVATION_MODE)

RUN_CONFIG = {
    "run_id": RUN_ID,
    "study_series_name": STUDY_SERIES_NAME,
    "observation_mode": OBSERVATION_MODE,
    "initial_checkpoint_drive": str(INITIAL_CHECKPOINT_DRIVE),
    "ground_thrust_penalty": GROUND_THRUST_PENALTY,
    "additional_training_episodes": ADDITIONAL_TRAINING_EPISODES,
    "num_envs": NUM_ENVS,
    "historical_score": {"eval_seed": 10_000, "episodes_per_world": 10},
    "robust_score": {"eval_seed": 10_000, "episodes_per_world": ROBUST_EPISODES_PER_WORLD},
    "replay_memory_capacity": REPLAY_MEMORY_CAPACITY,
    "training_config": TRAINING_CONFIG.__dict__,
}

RUN_CONFIG

In [ ]:
# cell: checkpoint-input; requires: run-config

RUN_PATHS = prepare_run_directory(LOCAL_RUN_ROOT, RUN_ID, initial_checkpoint=INITIAL_CHECKPOINT_DRIVE)
write_yaml(RUN_PATHS.config, RUN_CONFIG)

print(f"run: {RUN_PATHS.root}")
print(f"initial checkpoint: {RUN_PATHS.initial_checkpoint}")

## Train

In [ ]:
# cell: training-env; requires: run-config, checkpoint-input

training_env = make_reward_shaping_vector_env(
    TRAINING_FACTORY, NUM_ENVS, ground_thrust_penalty=GROUND_THRUST_PENALTY
)

In [ ]:
# cell: train-shaped-checkpoint; requires: training-env

trainer = VectorTrainer(
    training_env,
    seed=77,
    device=device,
    replay_memory_capacity=REPLAY_MEMORY_CAPACITY,
    hidden_size=TRAINING_CONFIG.hidden_size,
)
initial_metadata = load_q_net_checkpoint(trainer.q_net, RUN_PATHS.initial_checkpoint, device)
trainer.target_net.load_state_dict(trainer.q_net.state_dict())

training_result = trainer.train(TRAINING_CONFIG)
training_env.close()

training_summary = {
    "initial_checkpoint_metadata": initial_metadata,
    "trained_episodes": len(training_result.episode_returns),
    "env_steps": training_result.env_steps,
    "optimizer_updates": training_result.optimizer_updates,
    "mean_return": sum(training_result.episode_returns) / len(training_result.episode_returns),
    "last_50_mean_return": sum(training_result.episode_returns[-50:]) / min(50, len(training_result.episode_returns)),
}
write_yaml(RUN_PATHS.training_summary, training_summary)
save_q_net_checkpoint(
    trainer.q_net,
    RUN_PATHS.shaped_checkpoint,
    {"run_id": RUN_ID, "training_config": TRAINING_CONFIG.__dict__, **training_summary},
)

training_summary

## Evaluate

In [ ]:
# cell: historical-score; requires: train-shaped-checkpoint

historical_result = historical_score(q_net=trainer.q_net, make_envs=EVALUATION_FACTORY.evaluation_envs(), device=device)

print(f"historical_score: {historical_result.score:.1f}")
historical_result.world_scores

In [ ]:
# cell: robust-score; requires: train-shaped-checkpoint

robust_result = robust_score(
    q_net=trainer.q_net,
    make_envs=EVALUATION_FACTORY.evaluation_envs(),
    episodes_per_world=ROBUST_EPISODES_PER_WORLD,
    device=device,
)

print(f"robust_score: {robust_result.score:.1f}")
robust_result.world_scores

## Save artifacts

In [ ]:
# cell: save-run-artifacts; requires: historical-score, robust-score

write_eval_scores(RUN_PATHS.eval_scores, [historical_result, robust_result])

score_summary = {
    "historical_score": historical_result.score,
    "historical_world_scores": historical_result.world_scores,
    "robust_score": robust_result.score,
    "robust_world_scores": robust_result.world_scores,
    "historical_ground_side_thrust_steps": sum(row.ground_side_thrust_steps for row in historical_result.rows),
    "robust_ground_side_thrust_steps": sum(row.ground_side_thrust_steps for row in robust_result.rows),
}
drive_run_path = DRIVE_RUN_ROOT / RUN_ID
drive_run_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_PATHS.root, drive_run_path, dirs_exist_ok=True)

print(f"saved local: {RUN_PATHS.root}")
print(f"saved drive: {drive_run_path}")
score_summary